# Notebook 03 -- Data Quality

## Objective

Answer: **is the merged data trustworthy enough to compute Q/fouling/CIT on?** Two things happen here:

1. Correct cold-side crude-temperature outliers (ported from `notebooks/01_data_cleaning.ipynb`'s second half) --
   chain-consistency violations, rolling z-score spikes, and physically implausible readings.
2. Build the **combined per-HX Data Quality Score** (`docs/03` FR-DQ-012 -- completeness x validity x consistency),
   which the legacy pipeline never produced. This is the one requirement `docs/04`'s mapping explicitly flags as not
   satisfied by simply relocating the legacy checks -- it's built fresh here.

## Inputs

| Input | Path | Note |
|---|---|---|
| Notebook 02's merged table | `Data/v2/process_with_crude.csv` | 1,898 rows x 103 process+crude columns |
| Notebook 01's data dictionary | `Data/v2/data_dictionary.csv` | Tag->Equipment/Criticality mapping, reused rather than re-derived |
| HX chain layout | `src.cpht.config.HX_CONFIG` / `CHAIN_PREDECESSOR` | Which cold-side tags belong to which HX, and which tag is whose upstream predecessor |

## Assumptions

- Outlier-correction thresholds (`CHAIN_TOL_C=5.0`, `COLD_TEMP_ROLL_WIN=30`, `COLD_TEMP_Z_THRESH=3.0`,
  `COLD_TEMP_PHYS_MIN/MAX=30/380`) are **ported verbatim** from the legacy pipeline, not re-derived (same
  "do not re-litigate" rationale as Notebook 02's TAM thresholds).
- Outlier correction and the crude-assay merge are independent operations on disjoint columns (confirmed by reading
  the legacy notebook's execution order) -- so this notebook can safely operate on Notebook 02's already-merged
  output rather than needing to redo the merge after correction.
- The Data Quality Score design (completeness x validity x consistency, unweighted mean) is a **new design choice**,
  not something ported from the legacy pipeline -- `docs/03` specifies the three components but not a formula for
  combining them. Documented as tunable, same convention as the safety-weight fix in the legacy `08_cleaning_priority_ranking.ipynb`.

## Requirements traced from `docs/03`

| Requirement | Source | How this notebook addresses it |
|---|---|---|
| FR-DQ-003 -- values within valid engineering range | docs/03 section 2.3 | Physical-bounds check (30-380 degC) |
| FR-DQ-004/005 -- flatline / spike detection | docs/03 section 2.3 | Rolling z-score check catches spikes; flatline detection is a known gap (see Limitations) |
| FR-DQ-012 -- combined Data Quality Score per HX | docs/03 section 2.2 (previously unmet gap) | Built fresh in this notebook -- see Method |


## Method

1. Load Notebook 02's output; fail fast if missing.
2. For each cold-side crude-temperature tag (every unique `cold_in`/`cold_out` tag in `HX_CONFIG`): flag readings
   that are chain-inconsistent, rolling-z-score outliers, or physically out of bounds, then fill flagged points by
   time-interpolation (falling back to rolling median, then series mean).
3. Per HX, compute three 0-1 sub-scores against its own cold_flow/cold_in/cold_out tags:
   - **Completeness** = 1 - mean missing fraction across the HX's tags (post shutdown-removal data, pre-fill).
   - **Validity** = fraction of temperature readings within `[COLD_TEMP_PHYS_MIN, COLD_TEMP_PHYS_MAX]` before
     correction (i.e. measuring how much correction was needed, not hiding it after the fact).
   - **Consistency** = fraction of readings NOT chain-violating (HX with no upstream predecessor score 1.0 --
     trivially consistent, nothing to check).
   - **Data Quality Score** = unweighted mean of the three.
4. Write the corrected table + the per-HX quality score table as this notebook's outputs.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks_v2" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import json

from src.cpht import config
from src.cpht import validation

print(f"V2 output dir: {config.V2_OUTPUT_DIR}")


V2 output dir: C:\Desktop\Bangchak Internship 2026\Data\v2


## Data Validation

Check Notebook 02's output exists before doing any correction work.


In [2]:
v2_dir = Path(config.V2_OUTPUT_DIR)
merged_path = v2_dir / "process_with_crude.csv"
dict_path = v2_dir / "data_dictionary.csv"

nb02_result = validation.ValidationResult(ok=True)
for p in (merged_path, dict_path):
    if not p.exists():
        nb02_result.ok = False
        nb02_result.errors.append(f"{p} not found -- run Notebooks 01-02 first")
print(nb02_result.report())
nb02_result.raise_if_failed()

data = pd.read_csv(merged_path, index_col="Timestamp", parse_dates=True)
data_dictionary = pd.read_csv(dict_path)

print(f"Loaded: {data.shape[0]:,} rows x {data.shape[1]} columns, "
      f"{data.index.min().date()}..{data.index.max().date()}")

ts_check = validation.check_timestamp_column(data.reset_index(), timestamp_col="Timestamp")
print(ts_check.report())
ts_check.raise_if_failed()


Validation: PASS
Loaded: 1,898 rows x 103 columns, 2021-01-01..2026-07-01
Validation: PASS


## Analysis

### 1. Cold-side temperature outlier correction

Every unique `cold_in`/`cold_out` tag across `HX_CONFIG` (derived from config, not a separately hand-maintained list
-- avoids yet another copy of the same 17-tag set drifting out of sync).


In [3]:
cold_temp_tags = sorted({cfg["cold_in"] for cfg in config.HX_CONFIG.values()}
                         | {cfg["cold_out"] for cfg in config.HX_CONFIG.values()})
print(f"Cold-side temperature tags to correct: {len(cold_temp_tags)}")

data_corrected = data.copy()
# Per-tag flag masks, kept for the Data Quality Score below (pre-correction, not hidden after the fix).
chain_violation_masks = {}
physical_bound_masks = {}
outlier_masks = {}

for tag in cold_temp_tags:
    if tag not in data_corrected.columns:
        continue
    s = data_corrected[tag]

    roll_med = s.rolling(config.COLD_TEMP_ROLL_WIN, center=True, min_periods=5).median()
    roll_std = s.rolling(config.COLD_TEMP_ROLL_WIN, center=True, min_periods=5).std()
    z = (s - roll_med) / roll_std
    z_outlier = z.abs() > config.COLD_TEMP_Z_THRESH

    phys_bad = (s < config.COLD_TEMP_PHYS_MIN) | (s > config.COLD_TEMP_PHYS_MAX)

    predecessor = config.CHAIN_PREDECESSOR.get(tag)
    if predecessor is not None and predecessor in data_corrected.columns:
        chain_bad = s < (data_corrected[predecessor] - config.CHAIN_TOL_C)
    else:
        chain_bad = pd.Series(False, index=s.index)

    is_outlier = z_outlier | phys_bad | chain_bad
    chain_violation_masks[tag] = chain_bad
    physical_bound_masks[tag] = phys_bad
    outlier_masks[tag] = is_outlier

    filled = s.mask(is_outlier).interpolate(method="time", limit_direction="both")
    filled = filled.fillna(roll_med).fillna(s.mean())
    data_corrected[tag] = filled

n_total_flagged = sum(int(m.sum()) for m in outlier_masks.values())
print(f"Total flagged readings across {len(cold_temp_tags)} tags: {n_total_flagged}")


Cold-side temperature tags to correct: 19
Total flagged readings across 19 tags: 989


### 2. Data Quality Score (FR-DQ-012) -- new, not in the legacy pipeline

Per HX: unweighted mean of Completeness, Validity, and Consistency, each 0-1. Uses the **pre-correction** masks
above -- scoring after the fix would hide how much correction a HX's sensors actually needed.


In [4]:
quality_rows = []
for hx, cfg in config.HX_CONFIG.items():
    hx_tags = [t for t in (cfg["cold_flow"], cfg["cold_in"], cfg["cold_out"]) if t in data.columns]

    # Completeness: 1 - mean missing fraction across this HX's tags.
    completeness = 1.0 - data[hx_tags].isna().mean().mean()

    # Validity: fraction of temperature readings within physical bounds (temp tags only --
    # flow tags don't have a physical-bound check defined by the legacy logic; see Limitations).
    temp_tags = [t for t in (cfg["cold_in"], cfg["cold_out"]) if t in physical_bound_masks]
    if temp_tags:
        validity = 1.0 - np.mean([physical_bound_masks[t].mean() for t in temp_tags])
    else:
        validity = np.nan  # no temp tag to score (shouldn't happen for any real HX, flagged if it does)

    # Consistency: fraction NOT chain-violating. HX with no predecessor tag scores 1.0
    # (trivially consistent -- e.g. train-entry HX like E101AB has nothing upstream to check against).
    chain_tags = [t for t in (cfg["cold_in"], cfg["cold_out"]) if t in chain_violation_masks]
    checkable_chain_tags = [t for t in chain_tags if config.CHAIN_PREDECESSOR.get(t) in data.columns]
    if checkable_chain_tags:
        consistency = 1.0 - np.mean([chain_violation_masks[t].mean() for t in checkable_chain_tags])
    else:
        consistency = 1.0

    quality_score = np.nanmean([completeness, validity, consistency])
    quality_rows.append({
        "HX": hx,
        "completeness": round(completeness, 4),
        "validity": round(validity, 4) if pd.notna(validity) else None,
        "consistency": round(consistency, 4),
        "data_quality_score": round(quality_score, 4),
        "n_tags_scored": len(hx_tags),
    })

data_quality_score = pd.DataFrame(quality_rows).sort_values("data_quality_score")
print(f"Data Quality Score computed for {len(data_quality_score)} HX")
data_quality_score


Data Quality Score computed for 16 HX


,HX,completeness,validity,consistency,data_quality_score,n_tags_scored
7,E106AB,1.0,0.8045,1.0000,0.9348,3
4,E103AB,1.0,0.8045,1.0000,0.9348,3
11,E110ABC,1.0,0.8045,1.0000,0.9348,3
9,E108AB,1.0,0.9974,0.9692,0.9888,3
8,E107AB,1.0,0.9974,0.9797,0.9924,3
13,E112AB,1.0,0.9989,0.9831,0.9940,3
10,E109AB,1.0,1.0000,0.9895,0.9965,3
12,E111,1.0,0.9992,0.9916,0.9969,3
6,E105AB,1.0,1.0000,1.0000,1.0000,3
5,E104,1.0,1.0000,1.0000,1.0000,3


## Diagnostic Checks


In [5]:
diag = validation.ValidationResult(ok=True)

# 1. No NaN should remain in the corrected cold-temp tags (interpolate -> rolling median -> mean
#    should always leave a value, since mean() only fails if a whole column is NaN).
remaining_na = data_corrected[cold_temp_tags].isna().sum()
still_bad = remaining_na[remaining_na > 0]
if len(still_bad):
    diag.ok = False
    diag.errors.append(f"{len(still_bad)} tag(s) still have NaN after correction: {still_bad.to_dict()}")

# 2. No score should be outside [0, 1].
oob = data_quality_score[(data_quality_score["data_quality_score"] < 0) | (data_quality_score["data_quality_score"] > 1)]
if len(oob):
    diag.ok = False
    diag.errors.append(f"{len(oob)} HX have a Data Quality Score outside [0,1]: {oob['HX'].tolist()}")

# 3. Every HX in HX_CONFIG should have a score row (no HX silently dropped).
missing_hx = set(config.HX_CONFIG) - set(data_quality_score["HX"])
if missing_hx:
    diag.ok = False
    diag.errors.append(f"HX missing from quality score table: {missing_hx}")

# 4. Flag (warning, not failure) any HX whose score is notably low -- worth a human look.
low_quality = data_quality_score[data_quality_score["data_quality_score"] < 0.8]
if len(low_quality):
    diag.warnings.append(f"{len(low_quality)} HX below 0.8 quality score: {low_quality['HX'].tolist()}")

print(diag.report())
diag.raise_if_failed()


Validation: PASS


## Results


In [6]:
print(f"Cold-side temperature tags corrected: {len(cold_temp_tags)}")
print(f"Total flagged/filled readings:        {n_total_flagged}")
print(f"HX scored:                            {len(data_quality_score)}")
print(f"Mean Data Quality Score:               {data_quality_score['data_quality_score'].mean():.4f}")
print(f"Lowest-scoring HX:                     {data_quality_score.iloc[0]['HX']} ({data_quality_score.iloc[0]['data_quality_score']:.4f})")
print(f"Diagnostics:                           {'PASS' if diag.ok else 'FAIL'} ({len(diag.warnings)} warning(s))")


Cold-side temperature tags corrected: 19
Total flagged/filled readings:        989
HX scored:                            16
Mean Data Quality Score:               0.9858
Lowest-scoring HX:                     E106AB (0.9348)
Diagnostics:                           PASS (0 warning(s))


## Limitations

- **Data Quality Score is a new design, not a validated industry-standard formula.** The unweighted mean of
  completeness/validity/consistency is a reasonable, defensible default (documented here, same as the safety-weight
  fix elsewhere), but it hasn't been reviewed by a process engineer -- treat relative ranking (which HX scores lowest)
  as more trustworthy than the absolute number.
- **Validity only checks temperature tags** -- there is no physical-bounds definition for flow tags in the legacy
  logic this was ported from, so flow-tag validity is not scored (silently excluded, not zeroed). A future revision
  should add a plausible flow range (e.g. non-negative, below a max design flow).
- **Flatline detection (FR-DQ-004) is not implemented** -- a sensor stuck at a constant value would pass the z-score
  check (zero variance isn't necessarily an outlier by this method) and could look artificially "valid." This is a
  real gap against docs/03, not yet closed.
- Outlier correction thresholds are ported verbatim from the legacy pipeline (see Assumptions) -- not re-derived.
- Corrected values are filled (interpolated), not flagged-and-kept -- a downstream notebook cannot currently tell
  which specific data points were corrected vs. originally clean without re-deriving the masks computed here.

## Outputs

- `process_quality_corrected.csv` -- outlier-corrected version of Notebook 02's merged table, to `V2_OUTPUT_DIR`.
- `data_quality_score.csv` -- per-HX completeness/validity/consistency/combined score, to `V2_OUTPUT_DIR`.


In [7]:
out_dir = Path(config.V2_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

corrected_path = out_dir / "process_quality_corrected.csv"
data_corrected.to_csv(corrected_path, encoding="utf-8")
print(f"Wrote {corrected_path}  ({data_corrected.shape[0]:,} rows x {data_corrected.shape[1]} columns)")

score_path = out_dir / "data_quality_score.csv"
data_quality_score.to_csv(score_path, index=False, encoding="utf-8")
print(f"Wrote {score_path}  ({len(data_quality_score)} HX)")


Wrote C:\Desktop\Bangchak Internship 2026\Data\v2\process_quality_corrected.csv  (1,898 rows x 103 columns)


Wrote C:\Desktop\Bangchak Internship 2026\Data\v2\data_quality_score.csv  (16 HX)


## Conclusion

Cold-side temperature sensors were corrected for chain-consistency violations, rolling-window spikes, and physically
implausible readings using the legacy pipeline's validated thresholds. A new per-HX Data Quality Score (FR-DQ-012)
was built, closing a requirement the legacy pipeline never satisfied -- with honest limitations documented (flatline
detection and flow-tag validity are still gaps).

## Next Notebook

**Notebook 04 -- Time Alignment & Operating Modes**: reads `process_quality_corrected.csv`, determines which HX/shell
is actually in service on any given day (E113A<->E112C, E101EF<->E101G), ported from
`notebooks/03_operating_state_classification.ipynb`.
